#### 2. Base GRN input data preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys, shutil, importlib, glob
import celloracle as co
import gimmemotifs
import faulthandler
faulthandler.enable()
import gc
gc.collect()
import scanpy as sc
import anndata as ad
import muon as mu

from celloracle import motif_analysis as ma
from celloracle.utility import save_as_pickled_object
from tqdm.notebook import tqdm
%matplotlib inline

TSS annotation

In [ ]:
# Load scATAC-seq peak list.
peaks = pd.read_csv("/home/fuyq/GRN/single_cell/celloracle/pbmc10k/result/cellorcale/BaseGRN_input_data_preparation/all_peaks.csv", index_col=0)
peaks = peaks.x.values
# Load Cicero coaccessibility scores.
cicero_connections = pd.read_csv("/home/fuyq/GRN/single_cell/celloracle/pbmc10k/result/cellorcale/BaseGRN_input_data_preparation/cicero_connections.csv", index_col=0)
# # Annotate peaks with nearest TSS
ma.SUPPORTED_REF_GENOME
tss_annotated = ma.get_tss_info(peak_str_list=peaks, ref_genome="hg38")
# Integrate TSS info and cicero connections
integrated = ma.integrate_tss_peak_with_cicero(tss_peak=tss_annotated,
                                               cicero_connections=cicero_connections)
# Remove peaks with weak co-accessibility scores.
peak = integrated[integrated.coaccess >= 0.8]
peak = peak[["peak_id", "gene_short_name"]].reset_index(drop=True)
peak.to_csv("/home/fuyq/GRN/single_cell/celloracle/pbmc10k/result/cellorcale/BaseGRN_input_data_preparation/processed_peak_file.csv")

Motif scanning

In [ ]:
genome_installation = ma.is_genome_installed(ref_genome="hg38",
                                             genomes_dir="/data/fuyq/")
peaks = ma.check_peak_format(peaks, ref_genome, genomes_dir="/data/fuyq/cellorcale/")
tfi = ma.TFinfo(peak_data_frame=peaks,
                ref_genome=ref_genome,
                genomes_dir="/data/fuyq/")
tfi.scan(fpr=0.05,
         motifs=None,
         verbose=False)
tfi.reset_filtering()
tfi.filter_motifs_by_score(threshold=10)
tfi.make_TFinfo_dataframe_and_dictionary(verbose=False)
df = tfi.to_dataframe()
df.to_parquet("/data/fuyq/cellorcale/base_GRN_dataframe.parquet")